In [30]:
from typing import Optional
from dotenv import load_dotenv
load_dotenv()
import os
import logging as logger
from azure.core.credentials import AzureKeyCredential
from openai import AzureOpenAI, OpenAI
from app.utils.azure_ai_search_retriever import AzureAISearchRetriever, SearchCredential
from app.utils.rag_orchestrator import RAGOrchestrator
from typing import List
import base64
import openai

logger.getLogger().setLevel(logger.WARN)


In [31]:

def create_search_credential() -> SearchCredential:
    """Creates the appropriate Azure Search credential based on environment variables."""
    api_key = os.getenv("AZURE_SEARCH_API_KEY")
    if api_key:
        logger.info("Using Azure Search API Key credential.")
        return AzureKeyCredential(api_key)


def create_openai_client(purpose: str = "embedding") -> Optional[AzureOpenAI | OpenAI]:
    """Creates an AzureOpenAI client instance based on environment variables."""
    if purpose == "embedding":
        endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
        api_key = os.getenv("AZURE_OPENAI_API_KEY")
        api_version = os.getenv("AZURE_OPENAI_API_VERSION")
        log_prefix = "Embedding"
    elif purpose == "chat":
        endpoint = None
        api_version = None
        api_key = os.getenv("AZURE_OPENAI_CHAT_API_KEY", os.getenv("AZURE_OPENAI_API_KEY"))
        log_prefix = "Chat"
    else:
        raise ValueError(f"Invalid purpose for OpenAI client: {purpose}")

    try:
        if endpoint is None or api_version is None:
            client = OpenAI(
                api_key=api_key,
            )
        else:
            client = AzureOpenAI(
                azure_endpoint=endpoint,
                api_key=api_key,
                api_version=api_version,
            )
        # Perform a simple test if needed (e.g., list models), but might add latency
        logger.info(
            f"Azure OpenAI client for {log_prefix} initialized successfully (Endpoint: {endpoint}, API Version: {api_version}).")
        return client
    except Exception as e:
        logger.error(f"Failed to initialize Azure OpenAI {log_prefix} client: {e}", exc_info=True)
        return None  # Return None instead of raising, allow graceful failure if only one part is needed

def encode_file(file_path):
    with open(file_path, "rb") as file:
        return base64.b64encode(file.read()).decode('utf-8')


In [32]:
# --- Azure Search Configuration ---
search_service_name = os.getenv("AZURE_SEARCH_SERVICE_NAME")
index_name = os.getenv("AZURE_SEARCH_INDEX_NAME")
search_dns_suffix = os.getenv("AZURE_SEARCH_DNS_SUFFIX", "search.windows.net")

if not search_service_name or not index_name:
    raise ValueError("Missing required environment variables: AZURE_SEARCH_SERVICE_NAME and AZURE_SEARCH_INDEX_NAME")

search_service_endpoint = f"https://{search_service_name}.{search_dns_suffix}"
search_credential = create_search_credential()  # Handles API key, SPN, DefaultAzureCredential

# --- Azure OpenAI Configuration ---
# Embedding Client (Optional for Retriever if only doing text search)
openai_embedding_client = create_openai_client(purpose="embedding")
openai_embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")

# Chat Client (Required for Orchestrator)
openai_chat_client = create_openai_client(purpose="chat")
openai_chat_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

if not openai_chat_client or not openai_chat_deployment:
    raise ValueError(
        "Azure OpenAI Chat client configuration (Endpoint, Key, Version, Deployment) is required and incomplete.")

In [33]:
# --- Instantiate Retriever ---
retriever = AzureAISearchRetriever(
    search_service_endpoint=search_service_endpoint,
    index_name=index_name,
    search_credential=search_credential,
    openai_client=openai_embedding_client,  # Pass the client instance
    openai_embedding_deployment=openai_embedding_deployment  # Pass the deployment name
    # select_fields=None, # Use default
    # embedding_dimension=1536 # Use default
)
logger.info("AzureAISearchRetriever instantiated.")

# --- Instantiate Orchestrator ---
orchestrator = RAGOrchestrator(
    retriever=retriever,
    chat_client=openai_chat_client,  # Pass the client instance
    chat_deployment=openai_chat_deployment  # Pass the deployment name
    # system_prompt="Your custom system prompt here", # Optional
    # context_template="Source: {source}\nContent: {content}\n---\n", # Optional
)
logger.info("RAGOrchestrator instantiated.")

In [34]:
# --- Example Query ---
user_query = "Why do we need to do Business Process Re-engineering as a part of implementing an HIS/EHR? Note: Your answer must be in your own words."
logger.info(f"Answering query: '{user_query}'")

answer, sources = orchestrator.answer_query(
    user_query=user_query,
    top_k_retrieval=3,
    search_type="hybrid",  # Try hybrid search
    use_semantic_ranking=False  # Set to True if configured and desired
)

if answer:
    print("\n--- Answer ---")
    print(answer)
    print("\n--- Sources ---")
    if sources:
        for source in sources:
            print(f"- {source}")
    else:
        print("No sources were cited (or retrieval failed).")
else:
    print("\nFailed to get an answer.")


--- Answer ---
Business Process Re-engineering (BPR) is essential when implementing a Health Information System (HIS) or Electronic Health Record (EHR) because it ensures that the new system aligns with and improves organizational workflows, processes, and outcomes. Implementing an HIS/EHR is not just about adopting new technology; it requires rethinking and restructuring existing workflows to maximize the benefits of the system. 

For example, the adoption of an EHR led to significant improvements in workflow consistency, documentation quality, and coding accuracy. Uniform clinical workflows were established, which facilitated auditing and improved processes such as laboratory test tracking and order management. These improvements would have been challenging or impossible with previous systems, like paper charts or legacy EHRs. Additionally, issues like inefficient data entry workflows and problems with patient-entered data during registration highlighted the importance of redesignin

## Load Google Drive Files
Since our datasets live in Google Drive, we connect to our data source. This particular method assumes you have [Drive for Desktop](https://dl.google.com/drive-file-stream/GoogleDriveSetup.exe) installed on your computer and you are accessing a local path. For this POC, we focus on assignment 2.

In [35]:
class StudentResponse:
    rubric_path: str
    submission_path: str
    def __init__(self, rubric_path, submission_path):
        self.rubric_path = rubric_path
        self.submission_path = submission_path

In [36]:
# Initialize list to hold student responses
student_responses: List[StudentResponse] = []
# Relevant material list
relevant_material = r"I:\.shortcut-targets-by-id\1q2B2T_aTytCWO8SdBLnWpTNEQcPtXqE8\BU MET\cs581_quiz_and_assignment_data\Assignment 2 – EHR Functional Requirements Worksheet\Assignment2_relevant_material.pdf"
assignment_requirements = r"I:\.shortcut-targets-by-id\1q2B2T_aTytCWO8SdBLnWpTNEQcPtXqE8\BU MET\cs581_quiz_and_assignment_data\Assignment 2 – EHR Functional Requirements Worksheet\CS581 Assignment 2 HIS Clinical (EHR) Functional Requirements 2025 Spring 1.pdf"

In [37]:
# Define the base directory
base_dir = r"I:\.shortcut-targets-by-id\1q2B2T_aTytCWO8SdBLnWpTNEQcPtXqE8\BU MET\cs581_quiz_and_assignment_data\Assignment 2 – EHR Functional Requirements Worksheet\24fallmetcs581_m1 submissions and rubrics"

# Iterate through student directories
for student_number in os.listdir(base_dir):
    student_path = os.path.join(base_dir, student_number)

    if os.path.isdir(student_path):  # Ensure it's a directory
        rubric_path = os.path.join(student_path, "rubric.docx")
        submission_path = os.path.join(student_path, "submission.pptx")

        # Check if both expected files exist
        if os.path.exists(rubric_path) and os.path.exists(submission_path):
            student_responses.append(StudentResponse(rubric_path=rubric_path, submission_path=submission_path))

In [38]:
response = openai_chat_client.responses.create(
    model="gpt-4o",
    input=[
        {
            "role": "system",
            "content": [
                {
                    "type": "input_text",
                    "text": "Please analyze the attached file."
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "input_file",
                    "filename": os.path.basename(relevant_material),
                    "file_data": f"data:application/pdf;base64,{encode_file(relevant_material)}"
                }
            ]
        }
    ],
    text={
        "format": {
            "type": "text"
        }
    },
    reasoning={},
    tools=[],
    temperature=1,
    max_output_tokens=2048,
    top_p=1,
    store=True
)


In [46]:
from IPython.display import Markdown, display
display(Markdown(response.output[0].content[0].text))

### Virginia Women’s Center Case Study Overview

#### Organizational Overview
- **Location:** Richmond, Virginia
- **Services:** Comprehensive women’s health, including Ob/Gyn, female urology, pelvic surgery, psychology, nutrition, and genetics.
- **Staff:** 25 physicians and 12 mid-level providers at five clinical locations.
- **Specialized Services:** Digital mammography, bone densitometry, and ultrasonography.
- **Hospitals:** Affiliated with five local hospitals.
- **Payors:** Anthem, Aetna, United Healthcare, and others. Also participates in programs for uninsured and underinsured.

#### EHR Implementation Goals
- **Challenges:** Slow chart turnaround, high costs, and legibility issues.
- **Objectives:** Reduce costs, improve revenue cycle, enhance patient and staff satisfaction, and enable remote access for providers.

#### Implementation Strategy
- **Approach:** Phased implementation across locations over 14 months.
- **Staff Involvement:** 100% physician buy-in required, with a sequential approach for training.
- **Pilot Site:** Initial implementation at the smallest office.

#### Training and Support
- **EHR Team:** Comprised of clinical directors and physician champions who provided ongoing support and training.
- **Vendor Support:** Initial on-site training was provided with subsequent remote support.

#### Tools and Standards
- **Documentation:** Standardized forms and procedures across all locations.
- **Interfaces:** Established with labs and imaging systems for seamless integration.

#### Results and Benefits
- **Financial:** Initial $300,000 transcription cost savings; unexpected staffing ratio increase but productivity gains.
- **Operational:** 11% increase in patient visits, 13% increase in relative value units (RVUs).
- **Efficiency:** Reduction in charge entry staff needed due to electronic processing.
- **Quality of Care:** Improved tracking and patient safety measures due to electronic workflows.

#### Lessons Learned
- **Challenges:** Issues with patient-entered data and initial workflow designs.
- **Critical Success Factors:** Phased rollout, thorough vendor evaluation, and engaged staff.

#### Future Directions
- **Integration:** Aim for further integration with external health information systems for enhanced patient data access.
- **Patient Engagement:** Enhance patient interaction through secure messaging and online access to records.

### Key Takeaways
The Virginia Women’s Center’s transition to an EHR system illustrates a successful blend of technology adoption with focused training and staff engagement, resulting in improved workflows, financial performance, and patient care.